# Escape Behavior Analysis — 통합 파이프라인
**참조:** Bhatt et al., *Nature Neuroscience* (2024) plot_escape_response_panel 스타일

## 구성
| 패널 | 내용 |
|------|------|
| ax1 | Arena overhead — 궤적 + head/movement 화살표 + landmark |
| ax2 | Normalized distance to shelter (정규화 궤적) |
| ax3 | Speed time series (mean ± SEM) |
| ax4 | Normalized distance time series (mean ± SEM) |
| ax5 | cos(goal angle) time series (mean ± SEM) |
| 추가 | 12,000Hz vs 16,000Hz 비교, Heatmap |

> **단일 동물**: `USE_MULTI = False` → H5 + camera_log + sound_log 직접 입력  
> **다중 동물**: `USE_MULTI = True`  → 폴더 GUI로 여러 동물 선택

## 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.path import Path as MplPath
from scipy.ndimage import gaussian_filter
import json, math, os, glob, pickle
import tkinter as tk
from tkinter import filedialog
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401

plt.rcParams.update({
    'font.family': 'Arial',
    'font.size'  : 13,
    'axes.linewidth' : 1.0,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
})
print("✅ 라이브러리 로드 완료")

## 전역 설정

In [ ]:
# ── 모드 및 경로 설정 GUI ──────────────────────────────
def _setup_config():
    cfg = {}
    root = tk.Tk()
    root.title("분석 설정")
    root.attributes("-topmost", True)
    root.configure(bg='#F5F5F5')
    root.resizable(False, False)

    mode_var = tk.IntVar(value=0)   # 0=단일, 1=다중

    # ── 모드 선택 ──
    frm_mode = tk.LabelFrame(root, text=" 모드 선택 ", bg='#F5F5F5',
                              font=('Arial', 10, 'bold'), padx=10, pady=6)
    frm_mode.grid(row=0, column=0, padx=15, pady=(12, 4), sticky='ew')

    rb_single = tk.Radiobutton(frm_mode, text="단일 동물",
                                variable=mode_var, value=0,
                                bg='#F5F5F5', font=('Arial', 10))
    rb_multi  = tk.Radiobutton(frm_mode, text="다중 동물  (다음 셀에서 폴더 선택)",
                                variable=mode_var, value=1,
                                bg='#F5F5F5', font=('Arial', 10))
    rb_single.pack(side='left', padx=12)
    rb_multi.pack(side='left', padx=12)

    # ── 단일 동물 경로 ──
    frm_path = tk.LabelFrame(root, text=" 단일 동물 경로 ", bg='#F5F5F5',
                              font=('Arial', 10, 'bold'), padx=10, pady=6)
    frm_path.grid(row=1, column=0, padx=15, pady=4, sticky='ew')

    h5_var  = tk.StringVar(value=r"")
    roi_var = tk.StringVar(value=r"")
    cam_var = tk.StringVar(value=r"")
    snd_var = tk.StringVar(value=r"")
    ses_var = tk.StringVar(value=r"")
    out_var = tk.StringVar(value=r"")
    aid_var = tk.StringVar(value="")

    fields = [
        ("H5 파일",    h5_var,  "file", [("HDF5", "*.h5"), ("All", "*.*")]),
        ("ROI JSON",   roi_var, "file", [("JSON", "*.json"), ("All", "*.*")]),
        ("Camera Log", cam_var, "file", [("CSV", "*.csv"), ("All", "*.*")]),
        ("Event Log",  snd_var, "file", [("CSV", "*.csv"), ("All", "*.*")]),
        ("Session TS", ses_var, "file", [("CSV", "*.csv"), ("All", "*.*")]),
        ("출력 폴더",  out_var, "dir",  None),
        ("Animal ID",  aid_var, "text", None),
    ]

    def _browse(var, kind, title, ftypes):
        if kind == "file":
            p = filedialog.askopenfilename(parent=root, title=title, filetypes=ftypes)
        else:
            p = filedialog.askdirectory(parent=root, title=title)
        if p:
            var.set(p)

    for i, (label, var, kind, ftypes) in enumerate(fields):
        tk.Label(frm_path, text=label, bg='#F5F5F5', width=12, anchor='e',
                 font=('Arial', 9)).grid(row=i, column=0, padx=(4, 6), pady=3, sticky='e')
        tk.Entry(frm_path, textvariable=var, width=72,
                 font=('Consolas', 8)).grid(row=i, column=1, pady=3, sticky='ew')
        if kind in ("file", "dir"):
            tk.Button(frm_path, text="찾기", bg='#E0E0E0',
                      command=lambda v=var, k=kind, t=label, f=ftypes:
                          _browse(v, k, t + " 선택", f)
                      ).grid(row=i, column=2, padx=(4, 6), pady=3)

    def _toggle_paths():
        if mode_var.get() == 1:
            frm_path.grid_remove()
        else:
            frm_path.grid()
        root.update_idletasks()
        root.geometry("")

    rb_single.configure(command=_toggle_paths)
    rb_multi.configure(command=_toggle_paths)

    def _confirm():
        cfg['use_multi']   = mode_var.get() == 1
        cfg['h5_path']     = h5_var.get()
        cfg['roi_path']    = roi_var.get()
        cfg['camera_log']  = cam_var.get()
        cfg['event_log']   = snd_var.get()
        cfg['session_ts']  = ses_var.get()
        cfg['output_path'] = out_var.get()
        cfg['animal_id']   = aid_var.get()
        root.destroy()

    tk.Button(root, text="  확인  ", command=_confirm,
              bg='#2B7BBD', fg='white', font=('Arial', 10, 'bold'),
              relief='flat', padx=10, pady=6,
              ).grid(row=2, column=0, pady=12)

    root.update_idletasks()
    w  = root.winfo_reqwidth()
    h  = root.winfo_reqheight()
    sw = root.winfo_screenwidth()
    sh = root.winfo_screenheight()
    root.geometry(f"+{(sw - w) // 2}+{(sh - h) // 2}")
    root.mainloop()
    return cfg


cfg         = _setup_config()
USE_MULTI   = cfg['use_multi']
H5_PATH     = cfg.get('h5_path',     '')
ROI_PATH    = cfg.get('roi_path',    '')
CAMERA_LOG  = cfg.get('camera_log',  '')
EVENT_LOG   = cfg.get('event_log',   '')
SESSION_TS  = cfg.get('session_ts',  '')
OUTPUT_PATH = cfg.get('output_path', '')
ANIMAL_ID   = cfg.get('animal_id',   '')

print(f"✅ 모드: {'다중 동물' if USE_MULTI else '단일 동물'}")
if not USE_MULTI:
    print(f"   Animal ID : {ANIMAL_ID}")
    print(f"   H5        : {H5_PATH}")
    print(f"   ROI       : {ROI_PATH}")
    print(f"   Camera Log: {CAMERA_LOG}")
    print(f"   Event Log : {EVENT_LOG}")
    print(f"   Session TS: {SESSION_TS or '(미선택 → 영상 전체 사용)'}")
    print(f"   Output    : {OUTPUT_PATH}")

# ── 세션 구간 설정 ─────────────────────────────────────
# session_timestamps.csv 형식 (timestamp_s 는 camera_log 와 같은 절대 시계):
#   event,timestamp_s,local_time,duration_since_start_s
#   start,30101.905057,2026-07-16 08:21:41.905,
#   save ,31928.416903,2026-07-16 08:52:08.416,1826.512
USE_SESSION_WINDOW   = True                      # False → 영상 전체 사용
SESSION_START_EVENTS = ('start',)                # 세션 시작 이벤트 이름
SESSION_END_EVENTS   = ('save', 'end', 'stop')   # 세션 종료 이벤트 이름

# ── 분석 파라미터 ──────────────────────────────────────
LIKELIHOOD_THRESHOLD = 0.6   # DLC 신뢰도 컷오프
FPS       = 60               # 프레임레이트
PRE_SEC   = 0.1              # threat onset 이전 구간
POST_SEC  = 6                # threat onset 이후 구간 (데이터 로딩 기준)
PLOT_POST_SEC = 4.0          # 시계열 그래프 표시 범위 (초)
ARROW_STRIDE    = 1          # 화살표 간격 (프레임당 1개)
ARROW_DENSITY   = 1
ARROW_CM        = 5.0        # 화살표 길이 (ax1)
ARROW_NORM      = 0.08       # 화살표 길이 (ax2)
ARROW_WIDTH_MV  = 0.003     # movement direction 화살표 굵기
ARROW_WIDTH_HD  = 0.003      # head direction 화살표 굵기 (ax1)
ARROW_WIDTH_HD2 = 0.006     # head direction 화살표 굵기 (ax2)
# ── 색상 ───────────────────────────────────────────────
COLOR_TRACE    = "#A9A9A9"
COLOR_HEAD     = "#2F9BFF"
COLOR_SHELTER  = "#A2D9F8"
COLOR_SEM_FILL = "#A2D9F8"
COLOR_16000Hz = "#2B7BBD"
COLOR_12000Hz  = "#E8612C"

# ── Heatmap 파라미터 ──────────────────────────────────
BIN_SIZE_CM  = 3.0
SMOOTH_SIGMA = 1.5
SCATTER_STEP = 20
Z_MAX_TOP    = 3000
ZTICKS_TOP   = [0, 1000, 2000, 3000]
BOX_TOP      = [1, 1, 0.22]
BOX_BOT      = [1, 1, 0.25]
ELEV, AZIM   = 25, 35

# ── 공통 quiver 스타일 ─────────────────────────────────
QKWARGS = dict(
    pivot='tail', scale=1.0, scale_units='xy', angles='xy',
    headwidth=10, headlength=10, headaxislength=3,
)

# ── Heatmap colormap ───────────────────────────────────
CMAP_BLUE = LinearSegmentedColormap.from_list(
    'white_blue',
    ['#FFFFFF','#C6DCF0','#6EB0D8','#2B7BBD','#0D3F6B'], N=256
)

# ── threat type 매핑 (pkl 포맷용) ──────────────────────
THREAT_MAP = {3:'16,000Hz', 4:'12,000Hz', 'Sound1':'16,000Hz', 'Sound2':'12,000Hz'}

print("✅ 설정 완료")

## GUI 헬퍼 함수

In [ ]:
def _root():
    r = tk.Tk(); r.withdraw(); r.attributes("-topmost", True); return r

def pick_file(title, filetypes=None):
    r = _root()
    p = filedialog.askopenfilename(title=title,
        filetypes=filetypes or [("All","*.*")])
    r.destroy(); return p

def pick_files(title, filetypes=None):
    r = _root()
    p = filedialog.askopenfilenames(title=title,
        filetypes=filetypes or [("All","*.*")])
    r.destroy(); return list(p)

def pick_folders_loop(prompt):
    selected = []
    print(f"  폴더를 선택하세요. 없으면 '취소'.")
    while True:
        r = _root()
        f = filedialog.askdirectory(title=f"{prompt} ({len(selected)+1}번째)")
        r.destroy()
        if not f: break
        if f not in selected:
            selected.append(f)
            print(f"  ✅ {os.path.basename(f)}  ({len(selected)}개)")
    return selected

print("✅ GUI 헬퍼 로드 완료")

## 유틸리티 함수 (전처리 · 좌표 변환 · 정규화)

In [ ]:
# ── 보간 ──────────────────────────────────────────────
def interp_nan(x):
    x = np.asarray(x, float)
    idx = np.arange(x.size)
    isn = np.isnan(x)
    if np.all(isn) or np.all(~isn): return x
    x[isn] = np.interp(idx[isn], idx[~isn], x[~isn])
    return x

# ── Head 좌표 계산 (nose + R_ear + L_ear 무게중심) ────
def compute_head_coord(nose_x, nose_y, nose_l,
                       rear_x, rear_y, rear_l,
                       lear_x, lear_y, lear_l):
    nose_ok   = nose_l >= LIKELIHOOD_THRESHOLD
    rear_ok   = rear_l >= LIKELIHOOD_THRESHOLD
    lear_ok   = lear_l >= LIKELIHOOD_THRESHOLD
    both_ears = rear_ok & lear_ok

    hx = np.full(len(nose_x), np.nan)
    hy = np.full(len(nose_y), np.nan)

    all_ok = nose_ok & both_ears
    hx[all_ok] = (nose_x[all_ok] + rear_x[all_ok] + lear_x[all_ok]) / 3
    hy[all_ok] = (nose_y[all_ok] + rear_y[all_ok] + lear_y[all_ok]) / 3

    ears_only = ~nose_ok & both_ears
    hx[ears_only] = (rear_x[ears_only] + lear_x[ears_only]) / 2
    hy[ears_only] = (rear_y[ears_only] + lear_y[ears_only]) / 2

    return hx, hy

# ── DLC 좌표 전처리 (likelihood + arena범위) ──
def filter_and_interp(x, y, likelihood, arena_cx, arena_cy, arena_r):
    x = x.copy().astype(float)
    y = y.copy().astype(float)
    x[likelihood < LIKELIHOOD_THRESHOLD] = np.nan
    y[likelihood < LIKELIHOOD_THRESHOLD] = np.nan
    dist = np.sqrt((x-arena_cx)**2 + (y-arena_cy)**2)
    x[dist > arena_r*1.05] = np.nan
    y[dist > arena_r*1.05] = np.nan
    return interp_nan(x), interp_nan(y)

# ── px → cm 변환 ──────────────────────────────────────
def px_to_cm(x, y, arena_cx, arena_cy, px_per_cm):
    return (x-arena_cx)/px_per_cm, -((y-arena_cy)/px_per_cm)

# ── DLC h5 컬럼 파싱 ──────────────────────────────────
def get_xy(df, bodypart):
    cols = [c for c in df.columns if bodypart in c]
    xc = [c for c in cols if c.endswith('_x')][0]
    yc = [c for c in cols if c.endswith('_y')][0]
    lc = [c for c in cols if c.endswith('_likelihood')][0]
    return df[xc].values, df[yc].values, df[lc].values

# ── 타임스탬프 → 프레임 ───────────────────────────────
def ts_to_frame(ts, cam_df):
    idx = (cam_df['timestamp']-ts).abs().idxmin()
    fn  = cam_df.loc[idx,'frame_num']
    return int(fn - cam_df['frame_num'].iloc[0])

# ── shelter polygon 경계까지의 거리 (N×2 배열 입력) ──
def dist_to_poly_boundary(pts_cm, shelter_poly_cm):
    """
    각 점에서 shelter polygon 경계까지의 최소 거리.
    polygon 내부 점은 0 반환.
    """
    px = pts_cm[:, 0].astype(float)
    py = pts_cm[:, 1].astype(float)
    n_edge = len(shelter_poly_cm)
    min_d = np.full(len(px), np.inf)
    for i in range(n_edge):
        A = shelter_poly_cm[i].astype(float)
        B = shelter_poly_cm[(i + 1) % n_edge].astype(float)
        AB = B - A
        AB2 = float(np.dot(AB, AB))
        if AB2 < 1e-12:
            d = np.hypot(px - A[0], py - A[1])
        else:
            t = ((px - A[0]) * AB[0] + (py - A[1]) * AB[1]) / AB2
            t = np.clip(t, 0.0, 1.0)
            cx_e = A[0] + t * AB[0]
            cy_e = A[1] + t * AB[1]
            d = np.hypot(px - cx_e, py - cy_e)
        min_d = np.minimum(min_d, d)
    inside = MplPath(shelter_poly_cm).contains_points(pts_cm)
    min_d[inside] = 0.0
    return min_d

# ── 정규화 궤적 ───────────────────────────────────────
def norm_movement_trace(body_cm, shelter_cm, start_idx, end_idx,
                        shelter_poly_cm=None):
    """
    shelter_poly_cm 제공 시: y축 = shelter 경계까지의 거리 / 초기 경계 거리
    미제공 시: 기존 방식 (shelter 중심 기준)
    """
    seg = body_cm[start_idx:end_idx]
    if seg.shape[0] < 2: return None, None

    rel0 = seg[0] - shelter_cm
    d0_center = np.linalg.norm(rel0)
    if d0_center < 1e-6: return None, None

    # 탈출 방향 정렬 (초기 위치 → +y 방향)
    ang = math.atan2(rel0[1], rel0[0])
    dtheta = math.pi / 2 - ang
    c, s = math.cos(dtheta), math.sin(dtheta)
    R = np.array([[c, -s], [s, c]])
    rot = (seg - shelter_cm) @ R.T  # shelter 중심 기준 회전

    if shelter_poly_cm is not None:
        dists = dist_to_poly_boundary(seg, shelter_poly_cm)
        d0 = dists[0]
        if d0 < 1e-6: return None, None
        # x = 측면 이탈 / d0,  y = 경계까지 거리 / d0
        return rot[:, 0] / d0, dists / d0
    else:
        return rot[:, 0] / d0_center, rot[:, 1] / d0_center

# ── 스무딩 ────────────────────────────────────────────
def smooth(x, w=15):
    return np.convolve(x, np.ones(w)/w, mode='same')

# ── Arena 마스크 (heatmap용) ──────────────────────────
def make_arena_mask(xedges, yedges, cx, cy, r):
    xc = 0.5*(xedges[:-1]+xedges[1:])
    yc = 0.5*(yedges[:-1]+yedges[1:])
    XX, YY = np.meshgrid(xc, yc)
    return np.sqrt((XX-cx)**2+(YY-cy)**2) > r*1.02

print("✅ 유틸리티 로드 완료")

---
## 단일 동물 데이터 로드
> `USE_MULTI = False` 일 때 사용. 경로는 상단 **전역 설정** 셀에서 수정.
> `Session TS` 로 `session_timestamps.csv` 를 지정하면 세션 `start` ~ `save` 구간의
> 프레임만 잘라 사용하므로, 이후 모든 그래프가 영상 전체가 아닌 세션 구간만 그립니다.

In [ ]:
# ── 자극(trial onset) 이벤트 이름 ──────────────────────
STIM_EVENT_NAMES = ('Optogenetics', 'Threat')


def _load_event_log(event_log, cam_df):
    """
    이벤트 로그 파일을 자동 감지하여 (sound_type, timestamp) DataFrame 반환.

    지원 형식:
      1) 기존 sound_log: header 없음, (sound_type, timestamp_sec) 두 열
      2) timeline_events.csv: header 있음
         type,start_timestamp_s,end_timestamp_s,duration_s,frequency_Hz,event
         → STIM_EVENT_NAMES 에 해당하는 행만 추출.
           start_timestamp_s 는 camera_log 와 같은 절대 시계이므로 그대로 사용.
           sound_type 은 frequency_Hz 가 있으면 '8Hz' 형태, 없으면 event 이름.
    """
    # BOM(utf-8-sig) 포함 파일이 있어 열 이름이 깨지지 않도록 sig 사용
    with open(event_log, encoding='utf-8-sig') as f:
        first_line = f.readline()

    if 'start_timestamp_s' in first_line:
        # ── timeline_events.csv 형식 ──────────────────────
        ev_df = pd.read_csv(event_log, encoding='utf-8-sig')
        ev_df['event'] = ev_df['event'].astype(str).str.strip()
        stim = ev_df[ev_df['event'].isin(STIM_EVENT_NAMES)].copy()
        if stim.empty:
            raise ValueError(
                f"timeline_events 에 자극 이벤트({'/'.join(STIM_EVENT_NAMES)})가 없습니다: {event_log}\n"
                f"  파일에 있는 event: {sorted(ev_df['event'].unique())}")

        freq = pd.to_numeric(stim.get('frequency_Hz'), errors='coerce')
        snd_df = pd.DataFrame({
            'sound_type': [f"{fq:g}Hz" if pd.notna(fq) else ev
                           for fq, ev in zip(freq, stim['event'])],
            'timestamp' : stim['start_timestamp_s'].astype(float).values,
        })
    else:
        # ── 기존 sound_log 형식 ───────────────────────────
        snd_df = pd.read_csv(event_log, header=None,
                             names=['sound_type', 'timestamp'], dtype=str,
                             comment='#', skip_blank_lines=True)
        snd_df['timestamp'] = pd.to_numeric(snd_df['timestamp'], errors='coerce')
        snd_df = snd_df.dropna(subset=['timestamp']).copy()
        if snd_df.empty:
            raise ValueError(f"event_log 파일에 유효한 timestamp 데이터가 없습니다: {event_log}")
        snd_df['timestamp'] = snd_df['timestamp'].astype(float)

    return snd_df.sort_values('timestamp').reset_index(drop=True)


def _load_session_window(session_ts, cam_df, animal_id=""):
    """
    session_timestamps.csv → (t_start, t_end, 적용여부)

    형식:
        event,timestamp_s,local_time,duration_since_start_s
        start,30101.905057,2026-07-16 08:21:41.905,
        save ,31928.416903,2026-07-16 08:52:08.416,1826.512

    timestamp_s 는 camera_log 와 같은 절대 시계이므로 그대로 비교한다.
    파일이 없거나 USE_SESSION_WINDOW=False 이면 카메라 로그 전체 구간을 반환.
    """
    cam_t0 = float(cam_df['timestamp'].iloc[0])
    cam_t1 = float(cam_df['timestamp'].iloc[-1])

    if not USE_SESSION_WINDOW or not session_ts or not os.path.exists(session_ts):
        return cam_t0, cam_t1, False

    ts = pd.read_csv(session_ts, encoding='utf-8-sig')
    ts['event']       = ts['event'].astype(str).str.strip().str.lower()
    ts['timestamp_s'] = pd.to_numeric(ts['timestamp_s'], errors='coerce')
    ts = ts.dropna(subset=['timestamp_s'])

    t_start = ts.loc[ts['event'].isin(SESSION_START_EVENTS), 'timestamp_s']
    t_end   = ts.loc[ts['event'].isin(SESSION_END_EVENTS),   'timestamp_s']
    t0 = float(t_start.iloc[0])  if not t_start.empty else cam_t0
    t1 = float(t_end.iloc[-1])   if not t_end.empty   else cam_t1

    if t1 <= t0:
        print(f"  ⚠️  [{animal_id}] 세션 구간 비정상 "
              f"(start={t0:.1f}, end={t1:.1f}) → 영상 전체 사용")
        return cam_t0, cam_t1, False

    # 카메라 녹화 구간과의 교집합
    t0, t1 = max(t0, cam_t0), min(t1, cam_t1)
    if t1 <= t0:
        print(f"  ⚠️  [{animal_id}] 세션 구간과 카메라 구간이 겹치지 않음 → 영상 전체 사용")
        return cam_t0, cam_t1, False
    return t0, t1, True


def _session_frame_range(cam_df, t0, t1, n_frames):
    """세션 구간 [t0, t1] (절대 시각) → DLC 프레임 인덱스 구간 [i0, i1)"""
    ts_arr = cam_df['timestamp'].values
    rel    = cam_df['frame_num'].values - cam_df['frame_num'].iloc[0]
    a = int(np.searchsorted(ts_arr, t0, side='left'))
    b = int(np.searchsorted(ts_arr, t1, side='right')) - 1
    a = int(np.clip(a, 0, len(rel) - 1))
    b = int(np.clip(b, a, len(rel) - 1))
    i0 = int(np.clip(rel[a],     0,          max(n_frames - 1, 0)))
    i1 = int(np.clip(rel[b] + 1, i0 + 1,     n_frames))
    return i0, i1


def load_single_animal(h5_path, roi_path, camera_log, event_log, animal_id="",
                       session_ts=""):
    """
    DLC .h5 + roi_info.json + camera_log + sound_log/timeline_events
    (+ session_timestamps.csv) → 분석용 dict 반환

    session_timestamps.csv 가 주어지면 세션 start ~ save 구간의 프레임만 남기므로
    body_cm / head_cm / trial 인덱스 등 모든 산출물이 세션 구간으로 한정된다.
    """
    # ROI
    with open(roi_path, encoding='utf-8') as f: roi = json.load(f)
    acx, acy = roi['arena_center']
    ar        = roi['arena_radius']
    ppc       = roi['px_per_cm']
    ar_cm     = ar / ppc
    scx, scy  = roi['shelter_center']
    lm = []
    for k in ['landmark1_center','landmark2_center']:
        if k in roi:
            rk = k.replace('center','radius')
            lm.append({'xy': roi[k], 'r_cm': roi[rk]/ppc})

    # Shelter polygon (pixel → cm)
    if 'shelter_points' in roi:
        pts = np.array(roi['shelter_points'], dtype=float)
        sx_pts = (pts[:, 0] - acx) / ppc
        sy_pts = -((pts[:, 1] - acy) / ppc)
        shelter_poly_cm = np.column_stack([sx_pts, sy_pts])
    else:
        shelter_poly_cm = None

    # logs
    cam_df = pd.read_csv(camera_log, header=None,
                         names=['frame_num','type','timestamp'], dtype=str,
                         comment='#', skip_blank_lines=True)
    cam_df['frame_num'] = pd.to_numeric(cam_df['frame_num'], errors='coerce')
    cam_df['timestamp'] = pd.to_numeric(cam_df['timestamp'], errors='coerce')
    cam_df = cam_df.dropna(subset=['frame_num', 'timestamp']).copy()
    if cam_df.empty:
        raise ValueError(f"camera_log 파일에 유효한 frame_num/timestamp 데이터가 없습니다: {camera_log}")
    cam_df['frame_num'] = cam_df['frame_num'].astype(int)
    cam_df['timestamp'] = cam_df['timestamp'].astype(float)

    snd_df = _load_event_log(event_log, cam_df)

    # DLC h5
    df = pd.read_hdf(h5_path)
    df.columns = ['_'.join(col).strip() for col in df.columns.values]
    n_total = len(df)

    # ── 세션 구간([start, save])으로 프레임 crop ──────────
    sess_t0, sess_t1, sess_applied = _load_session_window(session_ts, cam_df, animal_id)
    i0, i1 = _session_frame_range(cam_df, sess_t0, sess_t1, n_total)
    df = df.iloc[i0:i1].reset_index(drop=True)
    if sess_applied:
        print(f"  [{animal_id}] 세션 구간 {sess_t1 - sess_t0:.1f}s  "
              f"→ frame {i0}~{i1}  ({i1 - i0}/{n_total} frames)")
    else:
        print(f"  [{animal_id}] 세션 구간 미적용 → 영상 전체 사용 ({n_total} frames)")

    # 자극 onset 프레임 (crop 된 구간 기준 인덱스)
    threat_frames, n_out = [], 0
    for _, row in snd_df.iterrows():
        rel = ts_to_frame(row['timestamp'], cam_df) - i0
        in_sess = (sess_t0 <= row['timestamp'] <= sess_t1)
        if not in_sess or not (0 <= rel < i1 - i0):
            n_out += 1
            continue
        threat_frames.append({'sound_type': str(row['sound_type']),
                               'rel_frame': rel,
                               'timestamp': row['timestamp']})
    if n_out:
        print(f"  [{animal_id}] 세션 구간 밖 자극 {n_out}개 제외")

    bc_x, bc_y, bc_l       = get_xy(df, 'body_center')
    nose_x, nose_y, nose_l = get_xy(df, 'nose')
    rear_x, rear_y, rear_l = get_xy(df, 'R_ear')
    lear_x, lear_y, lear_l = get_xy(df, 'L_ear')
    bc_x, bc_y = filter_and_interp(bc_x, bc_y, bc_l, acx, acy, ar)

    # head 좌표: nose + R_ear + L_ear 무게중심 (fallback 포함)
    head_x, head_y = compute_head_coord(
        nose_x, nose_y, nose_l,
        rear_x, rear_y, rear_l,
        lear_x, lear_y, lear_l)
    dist_h = np.sqrt((head_x - acx)**2 + (head_y - acy)**2)
    head_x[dist_h > ar * 1.05] = np.nan
    head_y[dist_h > ar * 1.05] = np.nan
    head_x = interp_nan(head_x)
    head_y = interp_nan(head_y)

    bc_xcm,   bc_ycm   = px_to_cm(bc_x,   bc_y,   acx, acy, ppc)
    head_xcm, head_ycm = px_to_cm(head_x, head_y, acx, acy, ppc)
    s_xcm, s_ycm = px_to_cm(scx, scy, acx, acy, ppc)
    lm_cm = [{'xy_cm': px_to_cm(*l['xy'], acx, acy, ppc), 'r_cm': l['r_cm']} for l in lm]

    body_cm    = np.c_[bc_xcm, bc_ycm]
    head_cm    = np.c_[head_xcm, head_ycm]
    shelter_cm = np.array([s_xcm, s_ycm])

    dist_to_shelter = np.sqrt((bc_xcm-s_xcm)**2+(bc_ycm-s_ycm)**2)
    dx = np.diff(bc_xcm, prepend=bc_xcm[0])
    dy = np.diff(bc_ycm, prepend=bc_ycm[0])
    speed = np.sqrt(dx**2+dy**2)*FPS

    sv_x = s_xcm-bc_xcm; sv_y = s_ycm-bc_ycm
    sm   = np.sqrt(sv_x**2+sv_y**2)
    hv_x = head_xcm-bc_xcm; hv_y = head_ycm-bc_ycm
    hm   = np.sqrt(hv_x**2+hv_y**2)
    cos_goal = np.where((sm>0)&(hm>0),
        (sv_x*hv_x+sv_y*hv_y)/(sm*hm), np.nan)

    speed_sm    = smooth(speed)
    cos_goal_sm = smooth(np.nan_to_num(cos_goal))

    # trial 추출
    pre  = int(PRE_SEC*FPS); post = int(POST_SEC*FPS)
    trajs, norm_dists, speeds, coss, valid_tfs = [], [], [], [], []
    for tf in threat_frames:
        fn = tf['rel_frame']
        s, e = fn-pre, fn+post
        if s < 0 or e >= len(bc_xcm): continue
        d0 = dist_to_shelter[fn]
        if d0 < 1e-6: d0 = 1.0
        nd = np.clip(dist_to_shelter[s:e]/d0, 0, 1.5)
        trajs.append((fn, s, e))
        norm_dists.append(nd)
        speeds.append(speed_sm[s:e])
        coss.append(cos_goal_sm[s:e])
        valid_tfs.append(tf)

    # threat_type 판별 (THREAT_MAP 에 없으면 sound_type 그대로 사용)
    threat_types = []
    for tf in valid_tfs:
        st  = tf['sound_type']
        key = int(st) if st.isdigit() else st
        threat_types.append(THREAT_MAP.get(key, st))

    breakdown = '  '.join(f"{t}={threat_types.count(t)}"
                          for t in sorted(set(threat_types)))
    print(f"  [{animal_id}] trials={len(trajs)}  {breakdown}")

    return {
        'animal_id'      : animal_id,
        'body_cm'        : body_cm,
        'head_cm'        : head_cm,
        'shelter_cm'     : shelter_cm,
        'shelter_poly_cm': shelter_poly_cm,
        'arena_r_cm'     : ar_cm,
        'lm_cm'          : lm_cm,
        'trajs'          : trajs,
        'norm_dists'     : norm_dists,
        'speeds'         : speeds,
        'coss'           : coss,
        'threat_types'   : threat_types,
        'valid_tfs'      : valid_tfs,
        'sess_t0'        : sess_t0,
        'sess_t1'        : sess_t1,
        'sess_applied'   : sess_applied,
        'frame_range'    : (i0, i1),
    }

if not USE_MULTI:
    print("단일 동물 로드 중...")
    animals = [load_single_animal(H5_PATH, ROI_PATH, CAMERA_LOG, EVENT_LOG,
                                  ANIMAL_ID, SESSION_TS)]
    print(f"✅ 로드 완료: {len(animals[0]['trajs'])}개 trial")
else:
    print("USE_MULTI=True → 다음 셀에서 폴더 선택")


In [ ]:
# ── 진단: 어떤 trial이 왜 탈락했는지 확인 ──────────────
def diagnose_trials(h5_path, roi_path, camera_log, event_log, animal_id="",
                    session_ts=""):
    cam_df = pd.read_csv(camera_log, header=None,
                         names=['frame_num','type','timestamp'])
    cam_df['frame_num'] = pd.to_numeric(cam_df['frame_num'], errors='coerce')
    cam_df['timestamp'] = pd.to_numeric(cam_df['timestamp'], errors='coerce')
    cam_df = cam_df.dropna(subset=['frame_num', 'timestamp'])
    cam_df['frame_num'] = cam_df['frame_num'].astype(int)
    cam_df['timestamp'] = cam_df['timestamp'].astype(float)

    snd_df = _load_event_log(event_log, cam_df)

    df = pd.read_hdf(h5_path)
    total_frames = len(df)

    cam_start = cam_df['timestamp'].iloc[0]
    cam_end   = cam_df['timestamp'].iloc[-1]
    cam_dur   = cam_end - cam_start

    sess_t0, sess_t1, sess_applied = _load_session_window(session_ts, cam_df, animal_id)
    i0, i1 = _session_frame_range(cam_df, sess_t0, sess_t1, total_frames)
    n_sess = i1 - i0

    pre  = int(PRE_SEC  * FPS)
    post = int(POST_SEC * FPS)

    print(f"=== [{animal_id}] 진단 ===")
    print(f"  camera_log  : {len(cam_df)} frames  ({cam_dur:.1f} sec)")
    print(f"  DLC H5      : {total_frames} frames")
    print(f"  cam_start ts: {cam_start:.3f}  cam_end ts: {cam_end:.3f}")
    if sess_applied:
        print(f"  세션 구간   : {sess_t0:.3f} ~ {sess_t1:.3f}  ({sess_t1 - sess_t0:.1f} sec)")
        print(f"                → frame {i0}~{i1}  ({n_sess} frames)")
    else:
        print(f"  세션 구간   : 미적용 (영상 전체 {total_frames} frames)")
    print(f"  pre={pre} frames  post={post} frames  (POST_SEC={POST_SEC}s x FPS={FPS})")
    print()

    for i, (_, row) in enumerate(snd_df.iterrows()):
        abs_ts  = row['timestamp']
        elapsed = abs_ts - sess_t0             # 세션 시작 기준 경과 시간(초)
        rel     = ts_to_frame(abs_ts, cam_df) - i0
        s, e    = rel - pre, rel + post
        in_cam  = (abs_ts >= cam_start) and (abs_ts <= cam_end)
        in_sess = (abs_ts >= sess_t0)  and (abs_ts <= sess_t1)
        ok      = in_sess and (s >= 0) and (e < n_sess)
        reason = []
        if not in_cam:  reason.append("abs_ts 카메라 범위 밖")
        if not in_sess: reason.append("abs_ts 세션 구간 밖")
        if s < 0:       reason.append(f"s={s} < 0")
        if e >= n_sess: reason.append(f"e={e} >= 세션 프레임수({n_sess})")
        status = "✅ OK" if ok else "❌ 탈락"
        print(f"  Stim {i+1} [{row['sound_type']}]: ts={abs_ts:.3f} "
              f"(+{elapsed:.1f}s)  rel_frame={rel}  s={s}  e={e}  {status}"
              + (f"  → {', '.join(reason)}" if reason else ""))

if not USE_MULTI:
    diagnose_trials(H5_PATH, ROI_PATH, CAMERA_LOG, EVENT_LOG, ANIMAL_ID, SESSION_TS)


---
## 다중 동물 데이터 로드
> `USE_MULTI = True` 일 때 사용.  
> 각 동물 폴더 안에 `.h5`, `roi_info.json`, `camera_log*.csv`, `sound_log*.csv` 가 있어야 합니다.
> `session_timestamps.csv` 가 있으면 자동으로 찾아 세션 구간만 사용합니다.

In [ ]:
def find_file(folder, pattern):
    """폴더에서 glob 패턴으로 첫 번째 파일 반환"""
    found = glob.glob(os.path.join(folder, '**', pattern), recursive=True)
    return found[0] if found else None

def load_multi_animals(folders):
    animals = []
    for folder in folders:
        aid = os.path.basename(folder)
        h5  = find_file(folder, '*.h5')
        roi = find_file(folder, 'roi_info.json')
        cam = find_file(folder, '*camera_log*.csv')
        snd = (find_file(folder, '*sound_log*.csv')
               or find_file(folder, 'timeline_events.csv')
               or find_file(folder, 'timeline_events_revised.csv'))
        ses = find_file(folder, 'session_timestamps*.csv')   # 없으면 영상 전체 사용
        if not all([h5, roi, cam, snd]):
            print(f"  ⚠️  {aid}: 필요 파일 누락 → 건너뜀")
            print(f"     h5={h5}  roi={roi}  cam={cam}  snd={snd}")
            continue
        if ses is None:
            print(f"  ⚠️  {aid}: session_timestamps.csv 없음 → 영상 전체 사용")
        try:
            a = load_single_animal(h5, roi, cam, snd, aid, ses or "")
            animals.append(a)
        except Exception as ex:
            print(f"  ❌ {aid}: {ex}")
    return animals

if USE_MULTI:
    print("동물 폴더를 선택하세요 (각 동물마다 1개 폴더).")
    folders = pick_folders_loop("동물 폴더 선택")
    if folders:
        animals = load_multi_animals(folders)
        print(f"\n✅ 총 {len(animals)}마리 로드 완료")
        total_trials = sum(len(a['trajs']) for a in animals)
        print(f"   총 trial: {total_trials}개")
    else:
        print("❌ 폴더 선택 취소")

---
## Figure 플롯 함수
> ax1(arena) + ax2(normalized) + ax3/4/5(time series)

In [ ]:
def plot_mean_sem(ax, data_list, time_axis, color=COLOR_HEAD, ylabel='', fontsize=16):
    valid = [d for d in data_list if len(d)==len(time_axis)]
    if not valid: return
    mean = np.nanmean(valid, axis=0)
    sem  = np.nanstd(valid, axis=0)/np.sqrt(len(valid))
    ax.fill_between(time_axis, mean-sem, mean+sem,
                    color=COLOR_SEM_FILL, alpha=0.4)
    ax.plot(time_axis, mean, color=color, linewidth=1.5)
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_ylabel(ylabel, fontsize=fontsize, labelpad=4)
    ax.set_xlim(-PRE_SEC, PLOT_POST_SEC)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


def plot_escape_response_panel(animal_data, threat_filter=None, title='', save_path=None):
    """
    Escape response panel 플롯.

    Parameters
    ----------
    animal_data  : load_single_animal 또는 merge_animals 반환값
    threat_filter: '16,000Hz' | '12,000Hz' | None (전체)
    """
    idx = [i for i, t in enumerate(animal_data['threat_types'])
           if threat_filter is None or t == threat_filter]
    if not idx:
        print(f"⚠ '{threat_filter}' 데이터 없음"); return None

    trajs       = [animal_data['trajs'][i]       for i in idx]
    norm_dists  = [animal_data['norm_dists'][i]  for i in idx]
    speeds      = [animal_data['speeds'][i]      for i in idx]
    coss        = [animal_data['coss'][i]        for i in idx]
    body_cm     = animal_data['body_cm']
    head_cm     = animal_data['head_cm']
    shelter_cm  = animal_data['shelter_cm']
    arena_r_cm  = animal_data['arena_r_cm']
    lm_cm       = animal_data.get('lm_cm', [])

    shelter_poly_cm = animal_data.get('shelter_poly_cm')
    _shelter_path   = MplPath(shelter_poly_cm) if shelter_poly_cm is not None else None

    pre  = int(PRE_SEC*FPS); post = int(POST_SEC*FPS)
    time_axis = np.linspace(-PRE_SEC, POST_SEC, pre+post)

    fig = plt.figure(figsize=(13, 6)); fig.patch.set_facecolor('white')
    gs  = gridspec.GridSpec(3, 3, figure=fig,
                            width_ratios=[2, 1.2, 0.9],
                            hspace=0.45, wspace=0.55)
    ax1 = fig.add_subplot(gs[:,0])
    ax2 = fig.add_subplot(gs[:,1])
    ax3 = fig.add_subplot(gs[0,2])
    ax4 = fig.add_subplot(gs[1,2])
    ax5 = fig.add_subplot(gs[2,2])

    # ── ax1: Arena overhead ──────────────────────────────
    ax1.set_aspect('equal'); ax1.set_facecolor('white'); ax1.axis('off')
    th = np.linspace(0, 2*np.pi, 400)
    ax1.plot(np.cos(th)*arena_r_cm, np.sin(th)*arena_r_cm,
             lw=1.5, color='black', zorder=0)

    for lm in lm_cm:
        ax1.add_patch(plt.Circle(lm['xy_cm'], lm['r_cm'],
            facecolor='gray', alpha=0.4,
            edgecolor='dimgray', linewidth=1.5, zorder=2))

    for fn, start, end in trajs:
        tx = body_cm[start:end, 0]; ty = body_cm[start:end, 1]
        hx_ = head_cm[start:end, 0]; hy_ = head_cm[start:end, 1]

        if _shelter_path is not None:
            inside  = _shelter_path.contains_points(np.column_stack([tx, ty]))
            reached = np.where(inside)[0]
        else:
            d_t     = np.sqrt((tx-shelter_cm[0])**2+(ty-shelter_cm[1])**2)
            reached = np.where(d_t < SHELTER_RADIUS_CM)[0]

        stop = reached[0] if len(reached) > 0 else len(tx)
        tx, ty = tx[:stop], ty[:stop]
        hx_, hy_ = hx_[:stop], hy_[:stop]

        if len(tx) == 0:
            continue

        ax1.plot(tx, ty, color=COLOR_TRACE, lw=1.5, alpha=0.5, zorder=3)

        dx_mv = np.diff(tx, prepend=tx[0]); dy_mv = np.diff(ty, prepend=ty[0])
        nrm = np.hypot(dx_mv, dy_mv); ok = nrm > 1e-6
        dx_mv[ok] /= nrm[ok]; dy_mv[ok] /= nrm[ok]
        dx_mv[~ok] = 0; dy_mv[~ok] = 0

        _all_s = np.arange(0, len(tx), max(1, int(ARROW_STRIDE)))
        idx_s  = _all_s[np.round(
            np.linspace(0, len(_all_s)-1, max(1, int(len(_all_s)*ARROW_DENSITY)))
        ).astype(int)]
        bx_s, by_s = tx[idx_s], ty[idx_s]
        hx_s, hy_s = hx_[idx_s], hy_[idx_s]
        dmv_xs, dmv_ys = dx_mv[idx_s], dy_mv[idx_s]

        if _shelter_path is not None:
            not_in_sh = ~_shelter_path.contains_points(np.column_stack([bx_s, by_s]))
        else:
            not_in_sh = np.sqrt((bx_s-shelter_cm[0])**2+(by_s-shelter_cm[1])**2) >= SHELTER_RADIUS_CM
        bx_s, by_s     = bx_s[not_in_sh], by_s[not_in_sh]
        hx_s, hy_s     = hx_s[not_in_sh], hy_s[not_in_sh]
        dmv_xs, dmv_ys = dmv_xs[not_in_sh], dmv_ys[not_in_sh]

        ax1.quiver(bx_s, by_s, dmv_xs*ARROW_CM, dmv_ys*ARROW_CM,
                   color=COLOR_TRACE, width=ARROW_WIDTH_MV, alpha=0.6, zorder=3, **QKWARGS)

        u = hx_s-bx_s; v = hy_s-by_s
        nrm = np.hypot(u,v); ok = nrm>1e-6
        u[ok] /= nrm[ok]; v[ok] /= nrm[ok]
        u[~ok] = 0; v[~ok] = 0
        ax1.quiver(bx_s, by_s, u*ARROW_CM, v*ARROW_CM,
                   color=COLOR_HEAD, width=ARROW_WIDTH_HD, alpha=0.8, zorder=4, **QKWARGS)

    if shelter_poly_cm is not None:
        ax1.add_patch(patches.Polygon(shelter_poly_cm, closed=True,
            facecolor=COLOR_SHELTER, alpha=0.9,
            edgecolor='#2C5F8A', linewidth=1.5, zorder=6))
        s_cx = float(np.mean(shelter_poly_cm[:, 0]))
        s_cy = float(np.mean(shelter_poly_cm[:, 1]))
    else:
        ax1.add_patch(Ellipse(xy=(shelter_cm[0], shelter_cm[1]),
            width=SHELTER_RADIUS_CM*3, height=SHELTER_RADIUS_CM*1.2,
            facecolor=COLOR_SHELTER, alpha=0.9, edgecolor='none', zorder=6))
        s_cx, s_cy = shelter_cm[0], shelter_cm[1]
    ax1.text(s_cx, s_cy, 'S', fontsize=12, fontweight='bold',
             color='#2C5F8A', ha='center', va='center', zorder=7)

    ax1.set_xlim(-arena_r_cm*1.2, arena_r_cm*1.2)
    ax1.set_ylim(-arena_r_cm*1.2, arena_r_cm*1.2)

    # 10 cm scale bar (bottom right)
    bar_x1 = arena_r_cm * 1.05
    bar_x0 = bar_x1 - 10.0
    bar_y  = -arena_r_cm * 1.05
    ax1.plot([bar_x0, bar_x1], [bar_y, bar_y],
             color='black', lw=2, solid_capstyle='butt', zorder=8)
    ax1.text((bar_x0 + bar_x1) / 2, bar_y - arena_r_cm * 0.04, '10 cm',
             ha='center', va='top', fontsize=9, color='black', zorder=8)

    ax1.legend(handles=[
        Line2D([0],[0], color=COLOR_HEAD,  lw=2.5, label='Head direction'),
        Line2D([0],[0], color=COLOR_TRACE, lw=2.5, label='Movement direction')
    ], loc='upper left', fontsize=16,
       bbox_to_anchor=(0.0, -0.02), bbox_transform=ax1.transAxes)

    # ── ax2: Normalized trajectories ────────────────────
    ax2.set_facecolor('white')

    for fn, start, end in trajs:
        x_n, y_n = norm_movement_trace(
            body_cm, shelter_cm, start, end, shelter_poly_cm)
        if x_n is None: continue

        if _shelter_path is not None:
            seg_body = body_cm[start:end]
            in_sh = _shelter_path.contains_points(seg_body)
            reached = np.where(in_sh)[0]
        else:
            dn = np.sqrt(x_n**2 + y_n**2)
            reached = np.where(dn < 0.1)[0]

        stop = reached[0] if len(reached) > 0 else len(x_n)
        x_n, y_n = x_n[:stop], y_n[:stop]

        if len(x_n) == 0:
            continue

        ax2.plot(x_n, y_n, color=COLOR_HEAD, alpha=0.7, lw=1.8, zorder=3)

        n_back = min(int(FPS * 0.25), len(x_n) - 1)
        if n_back > 0:
            u_s = x_n[-1] - x_n[-(n_back + 1)]
            v_s = y_n[-1] - y_n[-(n_back + 1)]
            nrm = np.hypot(u_s, v_s)
            if nrm > 1e-6:
                u_s /= nrm; v_s /= nrm
                tip_x, tip_y = float(x_n[-1]), float(y_n[-1])
                ax2.annotate('',
                             xy=(tip_x, tip_y),
                             xytext=(tip_x - u_s * 0.02, tip_y - v_s * 0.02),
                             xycoords='data', textcoords='data',
                             arrowprops=dict(
                                 arrowstyle='-|>, head_width=0.25, head_length=0.7',
                                 mutation_scale=16,
                                 lw=0,
                                 color=COLOR_HEAD,
                                 facecolor=COLOR_HEAD,
                             ), zorder=5)

    ax2.scatter([0],[0], s=400, color=COLOR_SHELTER,
                edgecolors='#2C5F8A', alpha=0.9, zorder=6)
    ax2.text(0, 0, 'S', fontsize=12, fontweight='bold',
             color='#2C5F8A', ha='center', va='center', zorder=7)

    ax2.set_ylim(1.05, -0.15); ax2.set_xlim(-0.6, 0.6)
    ax2.set_ylabel('Normalized distance to shelter', fontsize=20, labelpad=8)
    ax2.set_xticks([])
    ax2.tick_params(axis='y', labelsize=13)
    for sp in ['top','right','bottom']: ax2.spines[sp].set_visible(False)

    # ── ax3/4/5: 시계열 ──────────────────────────────────
    tcolor = COLOR_16000Hz if threat_filter=='16,000Hz' else (
             COLOR_12000Hz if threat_filter=='12,000Hz' else COLOR_HEAD)

    plot_mean_sem(ax3, speeds,     time_axis, color=tcolor,
                  ylabel='Speed\n(cm/s)', fontsize=16)
    plot_mean_sem(ax4, norm_dists, time_axis, color=tcolor,
                  ylabel='Norm. distance\nto shelter', fontsize=16)
    plot_mean_sem(ax5, coss,       time_axis, color=tcolor,
                  ylabel='cos\n(goal angle)', fontsize=16)

    for ax in [ax3, ax4, ax5]:
        ax.tick_params(axis='both', labelsize=13)

    ax4.set_ylim(0, 1.05); ax5.set_ylim(-0.2, 1.1)
    ax3.set_xticklabels([]); ax4.set_xticklabels([])
    ax5.set_xlabel('Time from\nescape onset (s)', fontsize=16)

    ttag = f' [{threat_filter}]' if threat_filter else ''
    n_tag = f'  n={len(trajs)} trials'
    plt.suptitle(f"{title}{ttag}{n_tag}", fontsize=15, y=1.01)

    fig.subplots_adjust(left=0.06, right=0.97, top=0.93, bottom=0.12)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"저장 완료: {save_path}")
    plt.show(); return fig

print("✅ plot_escape_response_panel 로드 완료")

---
## 다중 동물 데이터 병합 함수

In [ ]:
def merge_animals(animals):
    """
    여러 동물 dict를 하나로 병합.
    body_cm / head_cm은 연결(concatenate)하고 trajs 인덱스를 오프셋 조정.
    """
    if len(animals) == 1:
        return animals[0]

    merged = {
        'animal_id'      : '+'.join(a['animal_id'] for a in animals),
        'lm_cm'          : animals[0].get('lm_cm', []),
        'shelter_cm'     : animals[0]['shelter_cm'],
        'shelter_poly_cm': animals[0].get('shelter_poly_cm'),
        'arena_r_cm'     : animals[0]['arena_r_cm'],
        'trajs'          : [],
        'norm_dists'     : [],
        'speeds'         : [],
        'coss'           : [],
        'threat_types'   : [],
        'valid_tfs'      : [],
    }

    body_parts = []
    head_parts = []
    offset = 0

    for a in animals:
        n = len(a['body_cm'])
        body_parts.append(a['body_cm'])
        head_parts.append(a['head_cm'])
        for (fn, s, e) in a['trajs']:
            merged['trajs'].append((fn+offset, s+offset, e+offset))
        merged['norm_dists']  += a['norm_dists']
        merged['speeds']      += a['speeds']
        merged['coss']        += a['coss']
        merged['threat_types']+= a['threat_types']
        merged['valid_tfs']   += a['valid_tfs']
        offset += n

    merged['body_cm'] = np.vstack(body_parts)
    merged['head_cm'] = np.vstack(head_parts)
    return merged

print("✅ merge_animals 로드 완료")

---
## 실행 — Escape Response Panel
> 단일 동물이면 `animals[0]` 그대로, 다중 동물이면 `merge_animals(animals)` 후 플롯.

In [ ]:
data = animals[0]  # 단일 동물 데이터
aid  = data['animal_id']

# 전체
fig_all = plot_escape_response_panel(data, threat_filter=None,
                        title=aid,
                        save_path=OUTPUT_PATH if not USE_MULTI else None)

## Shelter 구간 설정

In [ ]:
def filter_trials_by_shelter(animal_data, trial_indices):
    if isinstance(trial_indices, slice):
        idx = list(range(*trial_indices.indices(len(animal_data['trajs']))))
    else:
        idx = list(trial_indices)
    filtered = {k: v for k, v in animal_data.items()
                if k not in ('trajs','norm_dists','speeds','coss','threat_types','valid_tfs')}
    filtered['trajs']        = [animal_data['trajs'][i]        for i in idx]
    filtered['norm_dists']   = [animal_data['norm_dists'][i]   for i in idx]
    filtered['speeds']       = [animal_data['speeds'][i]       for i in idx]
    filtered['coss']         = [animal_data['coss'][i]         for i in idx]
    filtered['threat_types'] = [animal_data['threat_types'][i] for i in idx]
    filtered['valid_tfs']    = [animal_data['valid_tfs'][i]    for i in idx]
    return filtered

# trial 번호로 지정
SHELTER_O_RANGE = (1, 5)   # 위협 1번~4번 = Shelter O
SHELTER_X_RANGE = (6, 6)  # 위협 5번~6번 = Shelter X

data_shelter_o = filter_trials_by_shelter(
    data, slice(SHELTER_O_RANGE[0]-1, SHELTER_O_RANGE[1]))
data_shelter_x = filter_trials_by_shelter(
    data, slice(SHELTER_X_RANGE[0]-1, SHELTER_X_RANGE[1]))

print(f"Shelter O: trial {SHELTER_O_RANGE[0]}–{SHELTER_O_RANGE[1]}  ({len(data_shelter_o['trajs'])}개)")
print(f"Shelter X: trial {SHELTER_X_RANGE[0]}–{SHELTER_X_RANGE[1]}  ({len(data_shelter_x['trajs'])}개)")

Shelter O

In [ ]:
fig_shelter_o = plot_escape_response_panel(
    data_shelter_o,
    threat_filter=None,
    title=f"{aid}  |  Shelter O  (trials {SHELTER_O_RANGE[0]}–{SHELTER_O_RANGE[1]})",
)

Shelter X

In [ ]:
fig_shelter_x = plot_escape_response_panel(
    data_shelter_x,
    threat_filter=None,
    title=f"{aid}  |  Shelter X  (trials {SHELTER_X_RANGE[0]}–{SHELTER_X_RANGE[1]})",
)

---
## 3D Trajectory & Occupancy

함수 정의

In [ ]:
def plot_figure_d(animal_data, save_path=None):
    body_cm    = animal_data['body_cm']
    shelter_cm = animal_data['shelter_cm']
    arena_r_cm = animal_data['arena_r_cm']
    trajs      = animal_data['trajs']
    aid        = animal_data['animal_id']

    s_x, s_y     = shelter_cm
    rot_angle    = np.pi/2 - np.arctan2(s_y, s_x)
    cr, sr       = np.cos(rot_angle), np.sin(rot_angle)
    def rotate(x, y): return cr*x - sr*y, sr*x + cr*y

    bc_xr, bc_yr           = rotate(body_cm[:,0], body_cm[:,1])
    shelter_xr, shelter_yr = rotate(s_x, s_y)
    print(f"arena_r = {arena_r_cm:.1f} cm | shelter @ ({shelter_xr:.1f}, {shelter_yr:.1f})")

    shelter_poly_cm = animal_data.get('shelter_poly_cm')
    if shelter_poly_cm is not None:
        spoly_xr, spoly_yr = rotate(shelter_poly_cm[:, 0], shelter_poly_cm[:, 1])
        _shelter_path_3d   = MplPath(np.column_stack([spoly_xr, spoly_yr]))

    pre_frames   = int(PRE_SEC  * FPS)
    post_frames  = int(POST_SEC * FPS)
    window       = pre_frames + post_frames
    total_frames = len(bc_xr)
    total_z      = len(trajs) * window
    z_scale      = Z_MAX_TOP / total_z if total_z > 0 else 1

    th     = np.linspace(0, 2*np.pi, 300)
    tick_r = int(arena_r_cm // 10) * 10

    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    from mpl_toolkits.mplot3d import proj3d as _proj3d

    fig = plt.figure(figsize=(6, 9))
    fig.patch.set_facecolor('white')
    gs  = gridspec.GridSpec(2, 1, height_ratios=[1.1, 1], hspace=0.05)

    # ── 위 패널: 3D trajectory ──────────────────────────
    ax3 = fig.add_subplot(gs[0], projection='3d')
    ax3.set_facecolor('white')

    idx_sc = np.arange(0, total_frames, SCATTER_STEP)
    z_gray = (idx_sc / total_frames) * Z_MAX_TOP
    ax3.scatter(bc_xr[idx_sc], bc_yr[idx_sc], z_gray,
                c='lightgray', s=0.5, alpha=0.25, depthshade=False)

    ax3.plot(np.cos(th)*arena_r_cm, np.sin(th)*arena_r_cm,
             np.zeros(300), 'k-', lw=1.0, alpha=0.5)

    if shelter_poly_cm is not None:
        n_pts = len(spoly_xr)
        wall_verts = []
        for i in range(n_pts):
            j = (i + 1) % n_pts
            wall_verts.append([
                (spoly_xr[i], spoly_yr[i], 0),
                (spoly_xr[j], spoly_yr[j], 0),
                (spoly_xr[j], spoly_yr[j], Z_MAX_TOP),
                (spoly_xr[i], spoly_yr[i], Z_MAX_TOP),
            ])
        ax3.add_collection3d(Poly3DCollection(wall_verts, alpha=0.15,
            facecolor='lightblue', edgecolor='#2C5F8A', linewidth=0.8))
    else:
        theta_c = np.linspace(0, 2*np.pi, 40)
        T_C, Z_C = np.meshgrid(theta_c, [0, Z_MAX_TOP])
        ax3.plot_surface(shelter_xr + SHELTER_RADIUS_CM*np.cos(T_C),
                         shelter_yr + SHELTER_RADIUS_CM*np.sin(T_C),
                         Z_C, alpha=0.12, color='lightblue',
                         linewidth=0, antialiased=False)

    for trial_idx, (fn, s, e) in enumerate(trajs):
        tx = bc_xr[s:e]; ty = bc_yr[s:e]
        z_off = trial_idx * window * z_scale
        tz    = np.arange(len(tx)) * z_scale + z_off

        if shelter_poly_cm is not None:
            inside  = _shelter_path_3d.contains_points(np.column_stack([tx, ty]))
            reached = np.where(inside)[0]
        else:
            dist_s  = np.sqrt((tx-shelter_xr)**2 + (ty-shelter_yr)**2)
            reached = np.where(dist_s < SHELTER_RADIUS_CM)[0]

        stop = reached[0] if len(reached) > 0 else len(tx)
        ax3.plot(tx[:stop], ty[:stop], tz[:stop],
                 color=COLOR_HEAD, lw=1.2, alpha=0.85)

    ax3.set_xlim(-arena_r_cm, arena_r_cm)
    ax3.set_ylim(-arena_r_cm, arena_r_cm)
    ax3.set_zlim(0, Z_MAX_TOP)
    ax3.set_xticks([-tick_r, 0, tick_r])
    ax3.set_yticks([-tick_r, 0, tick_r])
    ax3.set_zticks([1000, 2000, 3000])          # 0 제거
    ax3.set_xlabel(''); ax3.set_ylabel(''); ax3.set_zlabel('')
    ax3.tick_params(labelsize=11, pad=0)
    ax3.set_box_aspect(BOX_TOP)
    ax3.view_init(elev=ELEV, azim=AZIM)
    ax3.zaxis._axinfo['juggled'] = (1, 2, 0)

    # Shelter 레이블 — proj3d로 shelter 2D 투영 위치 계산 후 리더 라인 연결
    if shelter_poly_cm is not None:
        sh_cx3d = float(np.mean(spoly_xr))
        sh_cy3d = float(np.mean(spoly_yr))
    else:
        sh_cx3d = float(shelter_xr)
        sh_cy3d = float(shelter_yr)
    _x2d, _y2d, _ = _proj3d.proj_transform(sh_cx3d, sh_cy3d, 0, ax3.get_proj())
    ax3.annotate('Shelter',
                 xy=(_x2d, _y2d), xycoords='data',
                 xytext=(0.92, 0.38), textcoords='axes fraction',
                 fontsize=10, fontweight='bold', color='#1A3A5C',
                 ha='left', va='center', annotation_clip=False,
                 arrowprops=dict(arrowstyle='-', color='#1A3A5C', lw=1.0,
                                 shrinkA=3, shrinkB=5))

    # 10 cm 스케일바 — axes 좌표계로 3D 박스 아래 표시
    sb_line = Line2D([0.04, 0.18], [-0.06, -0.06],
                     transform=ax3.transAxes, lw=2.5, color='black', clip_on=False)
    ax3.add_artist(sb_line)
    ax3.text2D(0.11, -0.09, '10 cm', transform=ax3.transAxes,
               ha='center', va='top', fontsize=10, clip_on=False)

    # ── 아래 패널: 3D probability histogram ────────────
    ax2 = fig.add_subplot(gs[1], projection='3d')
    ax2.set_facecolor('white')

    bins_xy = np.arange(-arena_r_cm, arena_r_cm + BIN_SIZE_CM, BIN_SIZE_CM)
    H, xedges, yedges = np.histogram2d(bc_xr, bc_yr, bins=[bins_xy, bins_xy])
    H = H.T.astype(float)
    xc = (xedges[:-1] + xedges[1:]) / 2
    yc = (yedges[:-1] + yedges[1:]) / 2
    XX, YY  = np.meshgrid(xc, yc)
    outside = (XX**2 + YY**2) > arena_r_cm**2

    H_sm = gaussian_filter(np.nan_to_num(H), sigma=SMOOTH_SIGMA)
    H_sm[outside] = 0
    H_prob = H_sm / H_sm.sum() * 100
    H_plot = H_prob.copy()
    H_plot[(~outside) & (H_plot < 0.001)] = 0.001
    max_prob = H_prob.max()

    valid    = ~outside
    x_pos    = XX[valid] - BIN_SIZE_CM/2
    y_pos    = YY[valid] - BIN_SIZE_CM/2
    dz_vals  = H_plot[valid]
    norm_col = mcolors.Normalize(vmin=0, vmax=max_prob)
    colors   = cm.viridis(norm_col(dz_vals))

    ax2.bar3d(x_pos, y_pos, np.zeros_like(dz_vals),
              BIN_SIZE_CM, BIN_SIZE_CM, dz_vals,
              color=colors, alpha=0.9, shade=True, zsort='average')

    z_top = max(10, int(max_prob) + 1)
    ax2.set_xlim(-arena_r_cm, arena_r_cm)
    ax2.set_ylim(-arena_r_cm, arena_r_cm)
    ax2.set_zlim(0, z_top)
    ax2.set_xticks([-tick_r, 0, tick_r])
    ax2.set_yticks([-tick_r, 0, tick_r])
    ax2.set_zticks([5, 10])                     # 0 제거
    ax2.set_xlabel(''); ax2.set_ylabel(''); ax2.set_zlabel('')
    ax2.tick_params(labelsize=11, pad=0)
    ax2.set_box_aspect(BOX_BOT)
    ax2.view_init(elev=ELEV, azim=AZIM)
    ax2.zaxis._axinfo['juggled'] = (1, 2, 0)

    # 세로 컬러바 — 조금 더 굵게
    sm = cm.ScalarMappable(cmap='viridis', norm=norm_col)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax2, fraction=0.055, pad=0.12, shrink=0.45)
    cbar.set_label('Probability (%)', fontsize=8)
    cbar.set_ticks([0, 5, 10, int(max_prob)])
    cbar.ax.tick_params(labelsize=9)

    plt.suptitle(f"{aid}  |  Trajectory & Occupancy", fontsize=11, y=1.01)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"저장 완료: {save_path}")
    plt.show()
    return fig

print("✅ plot_figure_d 로드 완료")

 실행:

In [ ]:
fig_d = plot_figure_d(animals[0])

shelter O

In [ ]:
fig_d_shelter_o = plot_figure_d(data_shelter_o)

shelter X

In [ ]:
fig_d_shelter_x = plot_figure_d(data_shelter_x)

---
## 저장

In [ ]:
import tkinter as tk
from tkinter import filedialog

_r = tk.Tk(); _r.withdraw(); _r.attributes("-topmost", True)
SAVE_DIR = filedialog.askdirectory(title="저장 폴더 선택")
_r.destroy()

if not SAVE_DIR:
    print("❌ 폴더 선택 취소")
else:
    os.makedirs(SAVE_DIR, exist_ok=True)
    aid = data['animal_id']

    fig_all.savefig(os.path.join(SAVE_DIR, f'{aid}_escape_response_panel_all.pdf'),       dpi=300, bbox_inches='tight')
    fig_d.savefig(os.path.join(SAVE_DIR, f'{aid}_3D_Trajectory_Occupancy.pdf'),           dpi=300, bbox_inches='tight')
    fig_shelter_o.savefig(os.path.join(SAVE_DIR, f'{aid}_escape_response_panel_shelterO.pdf'), dpi=300, bbox_inches='tight')
    fig_shelter_x.savefig(os.path.join(SAVE_DIR, f'{aid}_escape_response_panel_shelterX.pdf'), dpi=300, bbox_inches='tight')
    fig_d_shelter_o.savefig(os.path.join(SAVE_DIR, f'{aid}_3D_shelterO.pdf'), dpi=300, bbox_inches='tight')
    fig_d_shelter_x.savefig(os.path.join(SAVE_DIR, f'{aid}_3D_shelterX.pdf'), dpi=300, bbox_inches='tight')
    print(f"✅ 저장 완료: {SAVE_DIR}")